# 02 - Codificador fundacional congelado (TimesFM 2.5)

Se evalúa si las representaciones de un modelo preentrenado de series de tiempo sirven como codificador del autoencoder de ciclos. Solo se entrenan una proyección lineal al espacio latente y el decodificador GRU refinado. Se incluyen dos controles: TimesFM con pesos aleatorios y la serie estandarizada aplanada.

El modelo preentrenado requiere descargar los pesos (`google/timesfm-2.5-200m-pytorch`, licencia Apache 2.0) y es considerablemente más rápido en GPU. Si no hay acceso a la descarga, la sección 3 lee los resultados guardados.

In [1]:
# Instalación para Colab o entorno local (una sola vez)
# %pip install -q torch pandas pyarrow scipy scikit-learn matplotlib
# %pip install -q git+https://github.com/google-research/timesfm.git
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cycle_manifold import SEEDS, SPECS, load_cycle_data
from cycle_manifold.data import find_cycles_file

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = load_cycle_data(find_cycles_file())
OUT = REPO / "results" / "foundation"
print({k: v.shape for k, v in data.X.items()})

{'train': (240, 101, 6), 'validation': (44, 101, 6), 'control_test': (47, 101, 6), 'case_external_test': (130, 101, 6)}


## 1. Verificación de los pesos

Se pronostican 32 puntos de una onda seno: con los pesos preentrenados el error es bajo (0.036 en la verificación original) y con pesos aleatorios es alto. Si la descarga no está disponible, la celda lo indica y el resto del cuaderno usa los resultados guardados.

In [2]:
from cycle_manifold.foundation import build_encoder, sanity_forecast_error

random_init = build_encoder("random")
print("Error de pronóstico, pesos aleatorios:   ", round(sanity_forecast_error(random_init.m), 3))
try:
    pretrained = build_encoder("timesfm")  # descarga los pesos
    print("Error de pronóstico, pesos preentrenados:", round(sanity_forecast_error(pretrained.m), 3))
except Exception as error:  # sin red o sin acceso a Hugging Face
    print("No se pudieron cargar los pesos preentrenados:", type(error).__name__)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error de pronóstico, pesos aleatorios:    1.877


No se pudieron cargar los pesos preentrenados: ProxyError


## 2. Experimento completo

Selecciona la configuración (bloque, agrupación, decaimiento de pesos) sobre validación con una semilla, y entrena cinco semillas por modelo. Equivale a `python -m cycle_manifold foundation-suite`. Para probar el flujo sin descargar el modelo, usar `dry_run=True` (pesos aleatorios).

In [3]:
RUN = False  # True para ejecutar; requiere los pesos y, idealmente, GPU
if RUN:
    from cycle_manifold.foundation_suite import run_suite
    table, chosen = run_suite(data, out_dir=OUT)

## 3. Resultados guardados

La tabla corresponde a la ejecución en GPU con los pesos preentrenados. El control de pesos aleatorios de esa tabla se ejecutó con las normas internas del modelo en cero, de modo que los bloques actuaban como identidad; la sección 4 lo reemplaza por el control corregido (normas en uno).

In [4]:
table = pd.read_csv(OUT / "resultados_modelo_fundacional.csv")
cols = ["mae_deg", "endpoint_jump_deg", "correction_gain", "typicality_gain", "intervention_deg"]
individual = table[table["seed"].astype(str) != "ensemble"]
individual.groupby(["tipo", "modelo"])[cols].agg(["mean", "std"]).round(3)

mae_deg        endpoint_jump_deg         \
                                  mean    std              mean    std   
tipo modelo                                                              
A    GRU mejorado (referencia)   4.113  0.049             0.866  0.106   
     Serie cruda                 3.687  0.111             1.396  0.054   
     TimesFM aleatorio           7.137  0.310             2.835  0.088   
     TimesFM preentrenado        7.186  0.222             2.546  0.145   
B    GRU mejorado (referencia)   4.827  0.074             0.567  0.045   
     Serie cruda                 4.462  0.057             1.284  0.127   
     TimesFM preentrenado       10.117  0.175             6.797  2.054   

                               correction_gain        typicality_gain         \
                                          mean    std            mean    std   
tipo modelo                                                                    
A    GRU mejorado (referencia)           0.189  0.012           0.326  0.007   
     Serie cruda                         0.011  0.021           0.181  0.013   
     TimesFM aleatorio                  -0.014  0.043           0.063  0.037   
     TimesFM preentrenado               -0.149  0.019           0.101  0.022   
B    GRU mejorado (referencia)           0.337  0.007           0.414  0.008   
     Serie cruda                         0.380  0.010           0.390  0.008   
     TimesFM preentrenado               -0.117  0.110          -0.049  0.028   

                               intervention_deg         
                                           mean    std  
tipo modelo                                             
A    GRU mejorado (referencia)            6.139  0.103  
     Serie cruda                          5.448  0.086  
     TimesFM aleatorio                    9.232  0.383  
     TimesFM preentrenado                 9.224  0.133  
B    GRU mejorado (referencia)            7.090  0.100  
     Serie cruda                          6.548  0.085  
     TimesFM preentrenado                11.474  0.045

In [5]:
import json
print(json.dumps({k: v for k, v in json.loads((OUT / "configuracion_elegida.json").read_text()).items() if k != "exploracion"}))
pd.DataFrame(json.loads((OUT / "configuracion_elegida.json").read_text())["exploracion"]).round(3)

{"layer": 9, "pooling": "meanlast", "wd": 0.001}


,bloque,agrupacion,decaimiento,dim,mae_deg,rmse_deg,endpoint_jump_deg,slope_jump_deg,vel_mae_deg,roughness_deg
0,9,meanlast,0.001,15372,7.431,9.906,2.652,4.607,0.868,0.094
1,9,mean,0.000,7692,7.595,10.081,3.428,4.577,0.861,0.091
2,9,mean,0.001,7692,7.626,10.043,3.139,4.265,0.863,0.082
3,9,meanlast,0.000,15372,7.655,10.162,2.765,4.322,0.883,0.092
4,-1,meanlast,0.001,15372,8.054,10.775,3.177,4.369,0.906,0.083
5,-1,mean,0.000,7692,8.118,10.748,3.504,5.022,0.890,0.085
6,-1,meanlast,0.000,15372,8.160,10.856,3.967,4.053,0.898,0.078
7,-1,mean,0.001,7692,8.276,10.819,3.113,4.529,0.904,0.076


## 4. Control de pesos aleatorios corregido (tipo A)

In [6]:
from scipy import stats

rand = json.loads((OUT / "control_random_active_A.json").read_text())
rand_ind = [r for r in rand if r["seed"] != "ensemble"]
pre = individual[(individual.tipo == "A") & (individual.modelo == "TimesFM preentrenado")].set_index(individual[(individual.tipo == "A") & (individual.modelo == "TimesFM preentrenado")]["seed"].astype(int))
rows = []
for m in cols:
    a = np.array([r[m] for r in rand_ind]); b = np.array([pre.loc[r["seed"], m] for r in rand_ind])
    rows.append({"metrica": m, "aleatorio": a.mean(), "preentrenado": b.mean(), "p (t pareada)": stats.ttest_rel(b, a)[1]})
pd.DataFrame(rows).set_index("metrica").round(3)

,aleatorio,preentrenado,p (t pareada)
metrica,,,
mae_deg,7.233,7.186,0.834
endpoint_jump_deg,2.738,2.546,0.185
correction_gain,-0.017,-0.149,0.004
typicality_gain,0.087,0.101,0.552
intervention_deg,9.336,9.224,0.373
